In [2]:

import pandas as pd

In [3]:
sales = pd.read_csv('../data/raw/sales_train_validation.csv')
calendar = pd.read_csv('../data/raw/calendar.csv')
prices = pd.read_csv('../data/raw/sell_prices.csv')
print("Sales data shape:", sales.shape)
print("Calendar data shape:", calendar.shape)
print("Prices data shape:", prices.shape)

Sales data shape: (30490, 1919)
Calendar data shape: (1969, 14)
Prices data shape: (6841121, 4)


In [4]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [5]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), str(6)
memory usage: 446.4 MB


In [6]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   date          1969 non-null   str  
 1   wm_yr_wk      1969 non-null   int64
 2   weekday       1969 non-null   str  
 3   wday          1969 non-null   int64
 4   month         1969 non-null   int64
 5   year          1969 non-null   int64
 6   d             1969 non-null   str  
 7   event_name_1  162 non-null    str  
 8   event_type_1  162 non-null    str  
 9   event_name_2  5 non-null      str  
 10  event_type_2  5 non-null      str  
 11  snap_CA       1969 non-null   int64
 12  snap_TX       1969 non-null   int64
 13  snap_WI       1969 non-null   int64
dtypes: int64(7), str(7)
memory usage: 215.5 KB


## 📌 Key Insight - Calendar Data

The calendar.csv has missing values in event columns:
- event_name_1 / event_type_1: only 162 out of 1969 days have values
- event_name_2 / event_type_2: only 5 out of 1969 days have values

**This is NOT a data quality issue.** It simply means most days are 
normal days with no special event. Empty = "no event happened that day".

No cleaning action needed for these columns.

In [7]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 208.8 MB


In [8]:
# Group products by category (HOBBIES, FOODS, HOUSEHOLD)
# and calculate total units sold across all years for each category
category_sales = sales.groupby('cat_id')[sales.columns[6:]].sum().sum(axis=1)
category_sales

cat_id
FOODS        45089939
HOBBIES       6124800
HOUSEHOLD    14480670
dtype: int64

## 📌 Key Insight - Category-wise Sales

Total units sold (2011-2016):
- FOODS: 45,089,939 units (74% of total)
- HOUSEHOLD: 14,480,670 units (24% of total)
- HOBBIES: 6,124,800 units (10% of total)

**FOODS is the dominant category**, selling ~3x more than HOUSEHOLD 
and ~7x more than HOBBIES. This makes sense - food items are 
necessities purchased frequently, while hobby items are discretionary 
purchases.

**Business implication:** Inventory planning should prioritize FOODS 
category to avoid stockouts, as it represents the bulk of demand.

In [9]:
# Group by store and calculate total units sold
store_sales = sales.groupby('store_id')[sales.columns[6:]].sum().sum(axis=1)
store_sales


store_id
CA_1     7698216
CA_2     5685475
CA_3    11188180
CA_4     4103676
TX_1     5595292
TX_2     7214384
TX_3     6089330
WI_1     5149062
WI_2     6544012
WI_3     6427782
dtype: int64

## 📌 Key Insight - Store-wise Sales

Top performer: CA_3 with 11.2 million units sold
Lowest performer: CA_4 with 4.1 million units sold

CA_3 sells nearly 3x more than CA_4, despite both being California 
stores. This suggests store-specific factors (size, location, local 
demographics) significantly impact sales volume - not just the state.

**Business implication:** CA_3's success factors should be studied 
and potentially replicated at underperforming stores like CA_4.

In [10]:
# Group by state and calculate total units sold
state_sales = sales.groupby('state_id')[sales.columns[6:]].sum().sum(axis=1)
state_sales

state_id
CA    28675547
TX    18899006
WI    18120856
dtype: int64

## 📌 Key Insight - State-wise Sales

Total units sold by state (2011-2016):
- CA: 28,675,547 units (44%)
- TX: 18,899,006 units (29%)
- WI: 18,120,856 units (27%)

California leads with significantly higher sales, partly due to having 
4 stores vs 3 stores in TX and WI. TX and WI perform similarly to 
each other.

**Note:** A fairer comparison would be average sales per store rather 
than total - CA's lead may partly just reflect having more stores.